# ДЗ 2. Функции и словари на настоящих данных
Срок: **до понедельника 19:00**.

**Сдача через GitHub** (инструкция «Git для домашних заданий»): сохраните ноутбук как `hw02.ipynb` в папку `hw02` репозитория `dsa-homework`, затем `git add hw02` → `git commit -m "hw02: ..."` → `git push`. Из Colab: Файл → Скачать → Скачать .ipynb. Файл `orders.csv` в репозиторий класть не нужно.

Файл `orders.csv` — заказы «Дала Маркет» за сентябрь и октябрь 2026. Работаем **без Pandas и NumPy**: только `csv`, функции, словари, множества.

- **[База]** обязательно для всех.
- **[Стандарт]** решайте, сколько получится.
- **[Челлендж]** для тех, кому легко.

**Правило AI:** можно спросить, *почему* не работает код, но нельзя просить решение.

## [Для всех] Чтение кода — без запуска
Для каждого фрагмента: **сначала** напишите в Markdown-ячейке ответ и одно предложение «почему», **потом** запустите и сравните. Ошибка в предсказании не снижает оценку — снижает отсутствие объяснения.
```python
# 1
r = {}
for c in ["A", "B", "A"]:
    r[c] = r.get(c, 0) + 1
print(r)

# 2
def f(x):
    x.append(1)
a = [0]; f(a); f(a); print(a)

# 3
s = {c.lower() for c in ["Almaty", "ALMATY", "Astana"]}; print(len(s))

# 4
d = {k: len(k) for k in ["ab", "abc", "ab"]}; print(d)

# 5
def g(n):
    if n > 0:
        print(n)
x = g(5); print(x)
```
На следующем занятии несколько человек объяснят вслух одну строку своего ДЗ.

Мои предсказания и объяснения:

1.
2.
3.
4.
5.

In [ ]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

In [ ]:
import csv
with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
print(len(rows), "строк")

## [База]
1. Напишите функции `to_num(s)` и `clean_city(s)` (6 городов, латиница и разный регистр → кириллица с заглавной).
2. Сколько заказов в каждой категории (`category`)? Отсортируйте по убыванию.
3. Для каждой категории посчитайте выручку оплаченных заказов и средний чек. Выведите таблицу f-строкой.
4. Сколько заказов в каждом месяце? Месяц — это `order_date[:7]`.

In [2]:
# База
import csv

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

print(list(rows[0].keys()))

def to_num(s):
    s = str(s).strip().replace(" ", "").replace("\xa0", "").replace(",", ".")
    if s == "":
        return None
    try:
        return float(s)
    except ValueError:
        return None

cities = {
    "almaty": "Алматы", "алматы": "Алматы",
    "astana": "Астана", "астана": "Астана",
    "shymkent": "Шымкент", "шымкент": "Шымкент",
    "karaganda": "Караганда", "караганда": "Караганда",
    "aktobe": "Актобе", "актобе": "Актобе",
    "atyrau": "Атырау", "атырау": "Атырау",
}

def clean_city(s):
    s = s.strip().lower()
    return cities.get(s, s.capitalize())

print(set(clean_city(r["city"]) for r in rows))

cat_count = {}
for r in rows:
    cat = r["category"]
    cat_count[cat] = cat_count.get(cat, 0) + 1

for cat, n in sorted(cat_count.items(), key=lambda x: x[1], reverse=True):
    print(cat, n)

rev = {}
cnt = {}
for r in rows:
    if r["status"] != "paid":
        continue
    amount = to_num(r["amount"])
    if amount is None:
        continue
    cat = r["category"]
    rev[cat] = rev.get(cat, 0) + amount
    cnt[cat] = cnt.get(cat, 0) + 1

print(f"{'Категория':<20}{'Заказов':>10}{'Выручка':>15}{'Ср. чек':>12}")
for cat in rev:
    print(f"{cat:<20}{cnt[cat]:>10}{rev[cat]:>15,.0f}{rev[cat] / cnt[cat]:>12,.0f}")

months = {}
for r in rows:
    m = r["order_date"][:7]
    months[m] = months.get(m, 0) + 1

for m in sorted(months):
    print(m, months[m])

['order_id', 'order_date', 'customer_id', 'city', 'category', 'amount', 'delivery_days', 'status']
{'Алматы', 'Шымкент', 'Караганда', 'Актобе', 'Атырау', 'Астана'}
Одежда 1529
Электроника 1185
Дом 1105
Красота 991
Книги 653
Категория              Заказов        Выручка     Ср. чек
2026-09 2782
2026-10 2681


## [Стандарт]
5. Выручка оплаченных по месяцам и изменение октября к сентябрю в процентах.
6. Доля отменённых заказов по каждому (чистому) городу, в процентах с одним знаком.
7. Топ-5 клиентов (`customer_id`) по выручке оплаченных заказов. Кто на первом месте и что с ним не так? Ответьте в Markdown-ячейке.
8. Сколько **уникальных** клиентов в каждом городе? Подсказка: словарь множеств.

In [ ]:
# Стандарт

## [Челлендж]
9. Найдите полные дубли строк: превратите каждую строку в `tuple(r.values())` и проверяйте через множество. Сколько их и в каком месяце?
10. Пересчитайте выручку по месяцам без дублей и без клиента из задачи 7. Как изменился процент из задачи 5?
11. У какого клиента больше всего заказов (после удаления дублей)? Сколько?
12. Напишите менеджеру 3–4 предложения: насколько на самом деле упала выручка и почему первый расчёт был неверным.

In [ ]:
# Челлендж